# Linkage vs Linkage Disequilibrium: An Interactive Exploration

## Making Sense of a Confusing Concept

**Authors:** Susama Kar & Dr. Alok Patel  
**Institution:** Department of Zoology, Kuchinda College, Sambalpur University

---

## 🤔 The Confusion

**Students often ask:**
- "Aren't linkage and linkage disequilibrium the same thing?"
- "Why does LD exist between genes on different chromosomes?"
- "How can LD 'decay' if genes don't move?"

**By the end of this notebook, you'll understand:**
1. 🧬 The critical difference between linkage and LD
2. 📊 How LD is measured (D, D', r²)
3. ⏰ Why LD decays over generations
4. 🧪 Applications to GWAS and population genetics
5. 🎚️ Interactive exploration of all concepts!

---

## 📚 Quick Definitions

### Linkage (Physical)
- **What:** Genes on the same chromosome
- **Cause:** Physical proximity
- **Changes?** NO - genes stay in same location
- **Time scale:** Permanent (millions of years)

### Linkage Disequilibrium (Statistical)
- **What:** Non-random association of alleles in a population
- **Cause:** Shared ancestry, selection, drift, migration
- **Changes?** YES - decays every generation
- **Time scale:** Temporary (generations to thousands of years)

### Key Insight
**Linkage is about LOCATION. LD is about ASSOCIATION.**


In [ ]:
# Setup and Imports

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, FancyBboxPatch
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, RadioButtons
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully!")
print("\n🎯 Ready to explore linkage and linkage disequilibrium!")

---

## 🧬 Part 1: Understanding the Difference

### The Setup

Consider two genes:
- **Gene A**: Alleles A₁ and A₂
- **Gene B**: Alleles B₁ and B₂

### Four Possible Haplotypes

| Haplotype | Alleles |
|-----------|----------|
| H₁ | A₁B₁ |
| H₂ | A₁B₂ |
| H₃ | A₂B₁ |
| H₄ | A₂B₂ |

### Linkage Equilibrium

**When there's NO association:**

Haplotype frequencies = product of allele frequencies

$$f(A_1B_1) = f(A_1) \times f(B_1)$$

### Linkage Disequilibrium

**When there IS association:**

$$f(A_1B_1) \neq f(A_1) \times f(B_1)$$

**Measured by D:**

$$D = f(A_1B_1) - f(A_1) \times f(B_1)$$

---

### 🎚️ Interactive: Create Your Own LD


In [ ]:
@interact(
    freq_A1B1=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.49, 
                         description='f(A₁B₁):', style={'description_width': 'initial'}),
    freq_A1B2=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.21, 
                         description='f(A₁B₂):', style={'description_width': 'initial'}),
    freq_A2B1=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.21, 
                         description='f(A₂B₁):', style={'description_width': 'initial'}),
    freq_A2B2=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.09, 
                         description='f(A₂B₂):', style={'description_width': 'initial'})
)
def explore_ld_basics(freq_A1B1, freq_A1B2, freq_A2B1, freq_A2B2):
    """
    Interactive LD calculator with visualization
    """
    # Normalize frequencies
    total = freq_A1B1 + freq_A1B2 + freq_A2B1 + freq_A2B2
    
    if abs(total - 1.0) > 0.01:
        print(f"⚠️ Warning: Frequencies sum to {total:.2f}, not 1.0")
        print("   Normalizing...\n")
        freq_A1B1 /= total
        freq_A1B2 /= total
        freq_A2B1 /= total
        freq_A2B2 /= total
    
    # Calculate allele frequencies
    freq_A1 = freq_A1B1 + freq_A1B2
    freq_A2 = freq_A2B1 + freq_A2B2
    freq_B1 = freq_A1B1 + freq_A2B1
    freq_B2 = freq_A1B2 + freq_A2B2
    
    # Calculate LD measures
    D = freq_A1B1 - (freq_A1 * freq_B1)
    
    # D' (normalized D)
    if D > 0:
        D_max = min(freq_A1 * freq_B2, freq_A2 * freq_B1)
    else:
        D_max = max(-freq_A1 * freq_B1, -freq_A2 * freq_B2)
    
    D_prime = D / D_max if D_max != 0 else 0
    
    # r² (correlation coefficient squared)
    denominator = freq_A1 * freq_A2 * freq_B1 * freq_B2
    r_squared = (D ** 2) / denominator if denominator > 0 else 0
    
    # Expected frequencies under LE
    exp_A1B1 = freq_A1 * freq_B1
    exp_A1B2 = freq_A1 * freq_B2
    exp_A2B1 = freq_A2 * freq_B1
    exp_A2B2 = freq_A2 * freq_B2
    
    # Visualization
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Haplotype frequencies (2x2 table)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.axis('tight')
    ax1.axis('off')
    
    haplotype_data = [
        ['', 'B₁', 'B₂', 'Total'],
        ['A₁', f'{freq_A1B1:.3f}', f'{freq_A1B2:.3f}', f'{freq_A1:.3f}'],
        ['A₂', f'{freq_A2B1:.3f}', f'{freq_A2B2:.3f}', f'{freq_A2:.3f}'],
        ['Total', f'{freq_B1:.3f}', f'{freq_B2:.3f}', '1.000']
    ]
    
    table = ax1.table(cellText=haplotype_data, cellLoc='center', 
                     bbox=[0.2, 0.0, 0.6, 1.0])
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)
    
    # Color code the cells
    for i in range(len(haplotype_data)):
        for j in range(len(haplotype_data[i])):
            if i == 0 or j == 0:  # Header
                table[(i, j)].set_facecolor('#E8E8E8')
                table[(i, j)].set_text_props(weight='bold')
            elif i < 3 and j < 3:  # Haplotype cells
                table[(i, j)].set_facecolor('#B3D9FF')
    
    ax1.set_title('Observed Haplotype Frequencies', fontsize=14, fontweight='bold', pad=20)
    
    # 2. Bar chart comparing observed vs expected
    ax2 = fig.add_subplot(gs[1, :])
    
    haplotypes = ['A₁B₁', 'A₁B₂', 'A₂B₁', 'A₂B₂']
    observed = [freq_A1B1, freq_A1B2, freq_A2B1, freq_A2B2]
    expected = [exp_A1B1, exp_A1B2, exp_A2B1, exp_A2B2]
    
    x = np.arange(len(haplotypes))
    width = 0.35
    
    bars1 = ax2.bar(x - width/2, observed, width, label='Observed', 
                    color='coral', alpha=0.8, edgecolor='black')
    bars2 = ax2.bar(x + width/2, expected, width, label='Expected (LE)', 
                    color='lightblue', alpha=0.8, edgecolor='black')
    
    ax2.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax2.set_title('Observed vs Expected Under Linkage Equilibrium', 
                 fontsize=13, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(haplotypes, fontsize=11, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add difference annotations
    for i, (obs, exp) in enumerate(zip(observed, expected)):
        diff = obs - exp
        y_pos = max(obs, exp) + 0.02
        color = 'red' if abs(diff) > 0.05 else 'green'
        ax2.text(i, y_pos, f'{diff:+.3f}', ha='center', 
                fontsize=9, fontweight='bold', color=color)
    
    # 3. LD Measures
    ax3 = fig.add_subplot(gs[2, 0])
    ax3.axis('off')
    
    ld_text = f"""
    D = {D:.4f}
    
    D' = {D_prime:.4f}
    
    r² = {r_squared:.4f}
    """
    
    ax3.text(0.1, 0.5, ld_text, fontsize=16, fontweight='bold',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax3.set_title('LD Measures', fontsize=13, fontweight='bold')
    
    # 4. D interpretation
    ax4 = fig.add_subplot(gs[2, 1])
    ax4.axis('off')
    
    if abs(D) < 0.001:
        interpretation = "Linkage Equilibrium\n\nAlleles are\nindependent"
        color = 'lightgreen'
    elif abs(D) < 0.05:
        interpretation = "Weak LD\n\nSlight association"
        color = 'lightyellow'
    elif abs(D) < 0.15:
        interpretation = "Moderate LD\n\nClear association"
        color = 'orange'
    else:
        interpretation = "Strong LD\n\nHigh association"
        color = 'coral'
    
    ax4.text(0.5, 0.5, interpretation, fontsize=14, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))
    ax4.set_title('Interpretation', fontsize=13, fontweight='bold')
    
    # 5. r² gauge
    ax5 = fig.add_subplot(gs[2, 2])
    
    theta = np.linspace(0, np.pi, 100)
    r = 1
    
    # Background arc
    ax5.plot(r * np.cos(theta), r * np.sin(theta), 'k-', linewidth=3)
    
    # r² indicator
    r2_angle = r_squared * np.pi
    ax5.plot([0, r * np.cos(r2_angle)], [0, r * np.sin(r2_angle)], 
            'r-', linewidth=4, marker='o', markersize=12)
    
    # Labels
    ax5.text(r * np.cos(0), r * np.sin(0) - 0.2, '0.0\nNo LD', 
            ha='center', fontsize=9, fontweight='bold')
    ax5.text(r * np.cos(np.pi/2), r * np.sin(np.pi/2) + 0.2, '0.5', 
            ha='center', fontsize=9, fontweight='bold')
    ax5.text(r * np.cos(np.pi), r * np.sin(np.pi) - 0.2, '1.0\nPerfect LD', 
            ha='center', fontsize=9, fontweight='bold')
    
    ax5.text(0, -0.5, f'r² = {r_squared:.3f}', ha='center', 
            fontsize=13, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
    
    ax5.set_xlim(-1.3, 1.3)
    ax5.set_ylim(-0.8, 1.3)
    ax5.set_aspect('equal')
    ax5.axis('off')
    ax5.set_title('r² (GWAS relevance)', fontsize=13, fontweight='bold')
    
    plt.show()
    
    # Print detailed explanation
    print("\n📊 Detailed Analysis:")
    print("=" * 70)
    print(f"\nAllele Frequencies:")
    print(f"  f(A₁) = {freq_A1:.3f},  f(A₂) = {freq_A2:.3f}")
    print(f"  f(B₁) = {freq_B1:.3f},  f(B₂) = {freq_B2:.3f}")
    
    print(f"\nLD Measures:")
    print(f"  D   = {D:.4f}  (Raw disequilibrium)")
    print(f"  D'  = {D_prime:.4f}  (Normalized, -1 to 1)")
    print(f"  r²  = {r_squared:.4f}  (For GWAS, 0 to 1)")
    
    print(f"\nInterpretation:")
    if abs(D) < 0.001:
        print("  ✅ Linkage Equilibrium: Alleles are independent")
        print("     Knowing one allele tells you nothing about the other")
    else:
        print(f"  🔗 Linkage Disequilibrium detected!")
        if D > 0:
            print("     → Positive LD: A₁B₁ and A₂B₂ are more common than expected")
            print("     → These alleles tend to be inherited together")
        else:
            print("     → Negative LD: A₁B₂ and A₂B₁ are more common than expected")
            print("     → These alleles tend to be on different haplotypes")
    
    print(f"\nr² Interpretation (for GWAS):")
    if r_squared > 0.8:
        print("  → Strong LD: SNPs are excellent proxies for each other")
    elif r_squared > 0.5:
        print("  → Moderate LD: SNPs capture some of each other's information")
    elif r_squared > 0.2:
        print("  → Weak LD: SNPs provide limited information about each other")
    else:
        print("  → Very weak/no LD: SNPs are nearly independent")
    
    print("\n" + "=" * 70)

### 💡 Try These Scenarios:

**1. Perfect Linkage Equilibrium:**
- f(A₁B₁) = 0.35, f(A₁B₂) = 0.15, f(A₂B₁) = 0.35, f(A₂B₂) = 0.15
- Notice: D = 0, all haplotypes match expectations!

**2. Perfect LD (Complete Association):**
- f(A₁B₁) = 0.7, f(A₁B₂) = 0.0, f(A₂B₁) = 0.0, f(A₂B₂) = 0.3
- Notice: Only two haplotypes exist! D' = 1.0, r² = 1.0

**3. Moderate LD:**
- f(A₁B₁) = 0.49, f(A₁B₂) = 0.21, f(A₂B₁) = 0.21, f(A₂B₂) = 0.09
- Notice: Deviation from expectations, but all haplotypes present

---

## ⏰ Part 2: LD Decay Over Generations

### Why Does LD Decay?

**Key concept:** Recombination **breaks apart** old haplotype combinations and **creates new** ones.

### The Decay Formula

$$D_t = D_0 \times (1 - r)^t$$

Where:
- $D_t$ = LD at generation t
- $D_0$ = Initial LD
- $r$ = Recombination fraction
- $t$ = Number of generations

### Critical Insights

1. **Linked genes** (small r): LD decays SLOWLY
2. **Unlinked genes** (r = 0.5): LD decays FAST
3. **Half-life:** $t_{1/2} = \ln(0.5) / \ln(1-r)$

---

### 🎚️ Interactive: Watch LD Decay


In [ ]:
@interact(
    initial_D=FloatSlider(min=0.01, max=0.25, step=0.01, value=0.20, 
                         description='Initial D:', style={'description_width': 'initial'}),
    recomb_frac=FloatSlider(min=0.0, max=0.50, step=0.01, value=0.10, 
                           description='Recomb. fraction (r):', style={'description_width': 'initial'}),
    generations=IntSlider(min=10, max=500, step=10, value=100, 
                         description='Generations:', style={'description_width': 'initial'}),
    show_halflife=Checkbox(value=True, description='Show half-life')
)
def simulate_ld_decay(initial_D, recomb_frac, generations, show_halflife):
    """
    Simulate and visualize LD decay over generations
    """
    # Calculate LD over time
    t = np.arange(0, generations + 1)
    D_t = initial_D * (1 - recomb_frac) ** t
    
    # Calculate half-life
    if recomb_frac > 0:
        half_life = np.log(0.5) / np.log(1 - recomb_frac)
    else:
        half_life = np.inf
    
    # Also calculate for comparison scenarios
    D_tight = initial_D * (1 - 0.01) ** t  # Tightly linked (1 cM)
    D_moderate = initial_D * (1 - 0.10) ** t  # Moderately linked (10 cM)
    D_unlinked = initial_D * (1 - 0.50) ** t  # Unlinked
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Main decay plot
    ax = axes[0]
    
    ax.plot(t, D_t, 'b-', linewidth=3, label=f'Your selection (r={recomb_frac:.2f})')
    
    # Add comparison lines
    ax.plot(t, D_tight, 'g--', linewidth=2, alpha=0.5, label='Tight linkage (r=0.01)')
    ax.plot(t, D_moderate, 'orange', linestyle='--', linewidth=2, alpha=0.5, 
           label='Moderate (r=0.10)')
    ax.plot(t, D_unlinked, 'r--', linewidth=2, alpha=0.5, label='Unlinked (r=0.50)')
    
    # Mark half-life
    if show_halflife and half_life < generations:
        D_half = initial_D / 2
        ax.axhline(y=D_half, color='gray', linestyle=':', linewidth=1, alpha=0.7)
        ax.axvline(x=half_life, color='gray', linestyle=':', linewidth=1, alpha=0.7)
        ax.plot(half_life, D_half, 'ro', markersize=10)
        ax.annotate(f't₁/₂ = {half_life:.1f} gen', 
                   xy=(half_life, D_half), xytext=(20, 20),
                   textcoords='offset points',
                   bbox=dict(boxstyle='round', fc='yellow', alpha=0.8),
                   arrowprops=dict(arrowstyle='->', lw=2),
                   fontsize=11, fontweight='bold')
    
    ax.set_xlabel('Generation', fontsize=13, fontweight='bold')
    ax.set_ylabel('D (Linkage Disequilibrium)', fontsize=13, fontweight='bold')
    ax.set_title('LD Decay Over Time', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, generations)
    ax.set_ylim(0, initial_D * 1.1)
    
    # Comparison of decay rates
    ax2 = axes[1]
    
    # Calculate % remaining at key generations
    checkpoints = [10, 50, 100, 200]
    checkpoints = [g for g in checkpoints if g <= generations]
    
    scenarios = {
        'Tight (r=0.01)': 0.01,
        'Moderate (r=0.10)': 0.10,
        'Loose (r=0.30)': 0.30,
        'Unlinked (r=0.50)': 0.50
    }
    
    x = np.arange(len(checkpoints))
    width = 0.2
    
    for i, (name, r_val) in enumerate(scenarios.items()):
        pct_remaining = [(1 - r_val) ** g * 100 for g in checkpoints]
        ax2.bar(x + i*width, pct_remaining, width, label=name, alpha=0.8)
    
    ax2.set_ylabel('% of Initial LD Remaining', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Generation', fontsize=12, fontweight='bold')
    ax2.set_title('Decay Rate Comparison', fontsize=13, fontweight='bold')
    ax2.set_xticks(x + 1.5*width)
    ax2.set_xticklabels(checkpoints)
    ax2.legend(fontsize=9, loc='upper right')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_ylim(0, 105)
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print("\n⏰ LD Decay Analysis:")
    print("=" * 70)
    print(f"\nParameters:")
    print(f"  Initial D = {initial_D:.3f}")
    print(f"  Recombination fraction (r) = {recomb_frac:.3f}")
    print(f"  Distance = {recomb_frac*100:.1f} cM (approximately)")
    
    print(f"\nDecay Kinetics:")
    if half_life != np.inf:
        print(f"  Half-life (t₁/₂) = {half_life:.1f} generations")
        print(f"  Time to 10% of initial LD = {-np.log(0.1)/np.log(1-recomb_frac):.1f} generations")
        print(f"  Time to 1% of initial LD = {-np.log(0.01)/np.log(1-recomb_frac):.1f} generations")
    else:
        print(f"  No recombination - LD persists forever!")
    
    # Check status at final generation
    final_D = D_t[-1]
    pct_remaining = (final_D / initial_D) * 100
    
    print(f"\nAfter {generations} generations:")
    print(f"  D = {final_D:.6f}")
    print(f"  {pct_remaining:.2f}% of initial LD remains")
    
    if pct_remaining > 50:
        print("  → Still substantial LD (genes are tightly linked)")
    elif pct_remaining > 10:
        print("  → Moderate LD remaining")
    elif pct_remaining > 1:
        print("  → Weak LD remaining")
    else:
        print("  → Nearly at linkage equilibrium")
    
    print("\n💡 Key Insight:")
    if recomb_frac < 0.05:
        print("  Tightly linked genes maintain LD for hundreds of generations!")
        print("  This is why we see LD blocks in the genome.")
    elif recomb_frac < 0.30:
        print("  Moderate linkage: LD decays but slowly.")
        print("  Useful for mapping - association signals extend to nearby variants.")
    else:
        print("  Loose/no linkage: LD decays rapidly!")
        print("  LD from population history disappears quickly.")
    
    print("\n" + "=" * 70)

### 🧪 Experiment:

**Question:** How long does LD persist?

**Try:**
1. r = 0.01 (1 cM, tightly linked) → LD persists 100+ generations
2. r = 0.10 (10 cM) → Half-life ~7 generations
3. r = 0.50 (unlinked) → Half-life = 1 generation!

**Insight:** This is why LD blocks exist - tightly linked variants maintain LD for thousands of generations!

---

## 🧬 Part 3: Linkage vs LD - The Critical Distinction

### Side-by-Side Comparison

| Aspect | Linkage | Linkage Disequilibrium |
|--------|---------|------------------------|
| **Definition** | Physical proximity on chromosome | Statistical association in population |
| **Cause** | Chromosome structure | Recent ancestry, selection, drift |
| **Measured by** | Recombination fraction (r) | D, D', r² |
| **Time scale** | Permanent | Temporary |
| **Can change?** | No (except chromosome rearrangement) | Yes (every generation) |
| **Between chromosomes?** | No | Yes! |
| **Affected by** | Physical distance | Physical distance AND population history |

### The Shocker: LD Can Exist Between Unlinked Genes!

**Reasons:**
1. **Population admixture** - Two populations mix, bringing different haplotypes
2. **Selection** - Favorable combinations increase together
3. **Genetic drift** - Random sampling in small populations
4. **Recent mutation** - New allele appears on specific haplotype background

**But:** LD between unlinked genes decays FAST (half-life = 1 generation)

---

### 🎚️ Interactive: Population Scenarios


In [ ]:
@interact(
    scenario=Dropdown(
        options=[
            ('Population Admixture', 'admixture'),
            ('Selection on Linked Genes', 'selection'),
            ('Genetic Drift (Founder Effect)', 'drift'),
            ('Random Mating (LE expected)', 'random')
        ],
        value='admixture',
        description='Scenario:',
        style={'description_width': 'initial'}
    ),
    recomb_rate=FloatSlider(min=0.0, max=0.50, step=0.05, value=0.10,
                           description='Recomb. rate:',
                           style={'description_width': 'initial'})
)
def explore_ld_scenarios(scenario, recomb_rate):
    """
    Explore different biological scenarios that create LD
    """
    # Define scenarios
    scenarios_data = {
        'admixture': {
            'title': 'Population Admixture',
            'description': 'Two populations with different allele frequencies mix',
            'pop1_haps': [0.8, 0.2, 0.0, 0.0],  # A1B1, A1B2, A2B1, A2B2 in Pop 1
            'pop2_haps': [0.0, 0.0, 0.2, 0.8],  # in Pop 2
            'mix_ratio': 0.5,
            'explanation': 'Population 1 has mostly A₁B₁, Population 2 has A₂B₂. When they mix, strong LD is created even for unlinked genes!'
        },
        'selection': {
            'title': 'Selection on Favorable Combinations',
            'description': 'A₁B₁ combination provides fitness advantage',
            'pop1_haps': [0.7, 0.1, 0.1, 0.1],
            'pop2_haps': None,
            'mix_ratio': None,
            'explanation': 'If A₁B₁ provides advantage, this combination increases in frequency, creating LD even if genes are unlinked!'
        },
        'drift': {
            'title': 'Genetic Drift (Founder Effect)',
            'description': 'Small founding population has extreme haplotype frequencies',
            'pop1_haps': [0.6, 0.3, 0.05, 0.05],
            'pop2_haps': None,
            'mix_ratio': None,
            'explanation': 'Random sampling in founders creates non-random associations by chance!'
        },
        'random': {
            'title': 'Random Mating (Equilibrium Expected)',
            'description': 'Large, randomly mating population',
            'pop1_haps': [0.35, 0.15, 0.35, 0.15],  # Close to LE
            'pop2_haps': None,
            'mix_ratio': None,
            'explanation': 'In large random-mating populations, LD should be minimal unless genes are linked!'
        }
    }
    
    data = scenarios_data[scenario]
    
    # Calculate initial haplotype frequencies
    if data['pop2_haps'] is not None:
        # Admixture scenario
        mix = data['mix_ratio']
        haps = [p1*mix + p2*(1-mix) for p1, p2 in zip(data['pop1_haps'], data['pop2_haps'])]
    else:
        haps = data['pop1_haps']
    
    # Calculate LD over generations
    generations = 50
    t = np.arange(0, generations + 1)
    
    # Initial D
    freq_A1 = haps[0] + haps[1]
    freq_B1 = haps[0] + haps[2]
    D0 = haps[0] - (freq_A1 * freq_B1)
    
    # LD decay
    D_t = D0 * (1 - recomb_rate) ** t
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Initial haplotype frequencies
    ax1 = axes[0, 0]
    hap_labels = ['A₁B₁', 'A₁B₂', 'A₂B₁', 'A₂B₂']
    colors_haps = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    bars = ax1.bar(hap_labels, haps, color=colors_haps, alpha=0.7, edgecolor='black', linewidth=2)
    ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax1.set_title('Initial Haplotype Frequencies', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    
    for bar, freq in zip(bars, haps):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{freq:.2f}', ha='center', fontweight='bold', fontsize=10)
    
    # 2. LD decay
    ax2 = axes[0, 1]
    ax2.plot(t, D_t, 'b-', linewidth=3)
    ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Generation', fontsize=12, fontweight='bold')
    ax2.set_ylabel('D', fontsize=12, fontweight='bold')
    ax2.set_title('LD Decay Over Time', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.fill_between(t, 0, D_t, alpha=0.3)
    
    # 3. Scenario description
    ax3 = axes[1, 0]
    ax3.axis('off')
    
    description_text = f"{data['title']}\n\n{data['description']}\n\n{data['explanation']}"
    ax3.text(0.5, 0.5, description_text, 
            ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8),
            wrap=True)
    
    # 4. LD measures
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # Calculate LD measures at t=0
    freq_A2 = haps[2] + haps[3]
    freq_B2 = haps[1] + haps[3]
    
    if D0 > 0:
        D_max = min(freq_A1 * freq_B2, freq_A2 * freq_B1)
    else:
        D_max = max(-freq_A1 * freq_B1, -freq_A2 * freq_B2)
    
    D_prime = D0 / D_max if D_max != 0 else 0
    
    denominator = freq_A1 * freq_A2 * freq_B1 * freq_B2
    r_squared = (D0 ** 2) / denominator if denominator > 0 else 0
    
    measures_text = f"""Initial LD Measures:
    
D = {D0:.4f}
D' = {D_prime:.4f}
r² = {r_squared:.4f}

After 10 generations:
D = {D_t[10]:.4f}
({(D_t[10]/D0)*100:.1f}% remaining)

Half-life: {-np.log(0.5)/np.log(1-recomb_rate) if recomb_rate > 0 else np.inf:.1f} gen
    """
    
    ax4.text(0.5, 0.5, measures_text, 
            ha='center', va='center', fontsize=12, fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    print(f"\n📖 Scenario: {data['title']}")
    print("=" * 70)
    print(f"\n{data['explanation']}")
    print(f"\nInitial D = {D0:.4f}")
    print(f"Recombination rate = {recomb_rate:.2f}")
    
    if recomb_rate == 0.5:
        print("\n⚠️ Genes are UNLINKED (different chromosomes or far apart)")
        print("   Yet LD exists! This shows LD ≠ linkage.")
        print("   LD will decay with half-life = 1 generation.")
    elif recomb_rate > 0:
        print(f"\nGenes are {recomb_rate*100:.0f} cM apart")
        half_life = -np.log(0.5)/np.log(1-recomb_rate)
        print(f"LD half-life = {half_life:.1f} generations")
    
    print("\n" + "=" * 70)

---

## 🎯 Summary & Key Takeaways

### What You've Learned:

1. ✅ **Linkage ≠ LD**
   - Linkage: Physical (where genes are)
   - LD: Statistical (how alleles associate)

2. ✅ **LD Measures**
   - D: Raw disequilibrium
   - D': Normalized (-1 to 1)
   - r²: For GWAS (0 to 1)

3. ✅ **LD Decays**
   - Formula: $D_t = D_0(1-r)^t$
   - Linked genes: Slow decay
   - Unlinked genes: Fast decay (1 gen half-life)

4. ✅ **LD Can Exist Between Unlinked Genes**
   - Admixture, selection, drift
   - But decays quickly

5. ✅ **Applications**
   - GWAS: r² > 0.8 for good proxy
   - Population genetics: Detect admixture, selection
   - Haplotype blocks: Explained by persistent LD

### The Bottom Line:

**Linkage tells us WHERE genes are.**  
**LD tells us WHICH alleles travel together in populations.**

Both are important, both are different!

---

## 📚 Further Exploration

### Questions to Think About:

1. Why do we care about LD in GWAS?
2. How does population history create LD?
3. What are "haplotype blocks" and why do they exist?
4. Can selection maintain LD between unlinked genes?

### Applications:

- **GWAS**: Use LD to find disease genes
- **Population genetics**: Infer history from LD patterns
- **Breeding**: Use LD for genomic selection
- **Evolution**: Detect selection from extended LD

---

**Authors:** Susama Kar & Dr. Alok Patel  
**Institution:** Kuchinda College, Sambalpur University  
**License:** CC BY 4.0  
**Repository:** github.com/The-Pattern-Hunter/principles-of-genetics-interactive
